#  Goal: build a model to predict whether a peptide is an activator or not in plant tissue.

## Prepare the data

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from tqdm import tqdm, notebook
notebook.tqdm().pandas()

# this is to set up the figure style
plt.style.use('default')
# Say, "the default sans-serif font is COMIC SANS"
mpl.rcParams['font.sans-serif'] = "Arial"
# Then, "ALWAYS use sans-serif fonts"
mpl.rcParams['font.family'] = "sans-serif"
mpl.rcParams['font.size']= 7
mpl.rcParams["svg.fonttype"] = "none"# to save text as actual text rather than vectorized shapes
mpl.rcParams['axes.linewidth'] = 0.5
mpl.rcParams['xtick.major.width'] = 0.5
mpl.rcParams['ytick.major.width'] = 0.5
mpl.rcParams['xtick.minor.width'] = 0.5
mpl.rcParams['ytick.minor.width'] = 0.5

%matplotlib inline
%config InlineBackend.figure_formats = ['svg']

First, we load the data.

The first file contains all of the features we will use as inputs and the mean measured activation values in both yeast and plants.

The second file tells us how far apart the n=2 replicates were for each peptide, information we will use to separate the means back into their original observed values.

In [ ]:
df_tf = pd.read_csv('Anya_plant_TF_all_data_annotated_features.csv')
residuals = pd.read_csv('all_data_6_2026_relaxed_with_residual.csv')

id_col = '#Feature'
target_plant = 'z_log_enrichment_mean'
target_yeast = 'PADI Score'


# Sort the two dataframes so they can recombine cleanly
df_tf = df_tf.sort_values(id_col)
residuals = residuals.sort_values(id_col)

# Downstream operations look at target, so we can choose what we model here
target = target_plant

# Sequence/annotation features start here; columns before this are metadata in newer CSVs.
original_features = list(
    df_tf.columns[df_tf.columns.get_loc('tile_context_disorder'):-1]
)

# residuals contains data for peptides not in df_tf
# Here we remove extraneous peptides and zip the two dataframes together
df_tf['replicate_residual_abs'] = residuals[residuals[id_col].isin(df_tf[id_col])]['replicate_residual_abs']

original_features

The data are not completely clean, so we need to remove any rows and columns that will compromise downstream modeling. That mostly means getting rid of rows/columns with NaN values in them, but it also means removing peptides active in yeast, since we are trying to judge what features differentiate a non-activator from a plant-only activator.

In [ ]:
print(df_tf.shape)

# Remove any line that is missing either target or replicate information
df_tf = df_tf[~(df_tf[target_plant].isna())]
df_tf = df_tf[~(df_tf[target_yeast].isna())]
df_tf = df_tf[~(df_tf['replicate_residual_abs'].isna())]
print(df_tf.shape)

# Print all features that contain nans, for scrutiny
print(df_tf.columns[df_tf.isna().any()].tolist())

# Remove features that contain nans
df_tf = df_tf.loc[:, df_tf.notna().all()]
print(df_tf.shape)

# ART demands this later
df_tf = df_tf.rename(columns = {id_col: 'Line Name'})
id_col = 'Line Name'

# Remove peptides active in yeast
df_tf = df_tf[df_tf[target_yeast] < 1]
print(df_tf.shape)

# Remove peptides with excessive measurement variance
df_tf = df_tf[df_tf['replicate_residual_abs'] < 1]
print(df_tf.shape)

# Remove from original_features any features that were deleted here
original_features = [
    feature
    for feature in original_features
    if feature in df_tf.columns and pd.api.types.is_numeric_dtype(df_tf[feature])
]
len(original_features)

Since we are working with replicates (n=2), we need to be able to keep track of which instances come from the same peptide. We don't want the replicates to ever occur on opposite side of a cross validation fold, since that would leak data and give us an artificially high score.

In [ ]:
def label_replicates(column):
    accum = []
    for value in column:
        num = 1
        value_tagged = f'{value}-R{num}'
        while value_tagged in accum:
            num += 1
            value_tagged = f'{value}-R{num}'
        accum.append(value_tagged)
    return accum    

We will set aside 20% of data, chosen at random and keeping replicates together, so that we can properly judge our model at the end of this pipeline. df_withheld will hold that set-aside data.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

df_tf['group'] = df_tf[id_col]

gss = GroupShuffleSplit(n_splits=2, test_size=0.2, random_state=2)

train_index, test_index = next(gss.split(df_tf, groups=df_tf['group']))

df_tf_train = df_tf.iloc[train_index]
df_withheld = df_tf.iloc[test_index]

df_tf = df_tf_train

print(df_withheld.shape)
print(df_tf.shape)
df_tf

In [ ]:
plt.hist(df_tf[target_plant], bins=list(range(-8, 8)))
plt.title('Training Data Distribution')
plt.xlabel(target_plant)
plt.ylabel('count')

Since we know the difference between the high and low measurements, we can simply add half the difference and subtract half the difference from the mean to get the original high and low values respectively.

In [ ]:
df_tf['enrichment_low'] = df_tf[target_plant] - df_tf['replicate_residual_abs'] / 2
df_tf['enrichment_high'] = df_tf[target_plant] + df_tf['replicate_residual_abs'] / 2
df_tf

Visually, our test data and training data should have a similarly shaped distribution with a differently scaled y-axis. That is indeed what we see here, so we don't need to worry.

In [ ]:
plt.hist(df_withheld[target_plant], bins=list(range(-8, 8)))
plt.title('Test Data Distribution')
plt.xlabel(target_plant)
plt.ylabel('count')

Here, we will create two copies of df_tf, one with the high values and one with the low values, so we can then stack them together and create our final training data set. We do not do the same with the test data since our overall goal is to predict the labels, and those are based on the mean.

In [ ]:
df_high = df_tf[original_features]
df_high.insert(0, target_plant, df_tf['enrichment_high'])
df_high.insert(0, 'Line Name', df_tf['Line Name'])
df_high

In [ ]:
df_low = df_tf[original_features]
df_low.insert(0, target_plant, df_tf['enrichment_low'])
df_low.insert(0, id_col, df_tf[id_col])
df_low

In [ ]:
df_tf = pd.concat([df_high, df_low], axis=0)
df_tf = df_tf.reset_index()
df_tf.insert(2, 'group', df_tf[id_col])
df_tf[id_col] = label_replicates(df_tf[id_col])

# Re-sync feature list after concat (avoids stale string cols from an old original_features).
_non_feature = {id_col, 'Line Name', target, 'group', 'index', 'level_0'}
original_features = [
    c for c in df_tf.columns
    if c not in _non_feature and pd.api.types.is_numeric_dtype(df_tf[c])
]

df_tf

## Modeling: Use an ensemble of XGBoost regressors

XGBoost regressors train an ensemble of decision trees, each using a subset of the features and a subset of the instances, providing a robust overall model for predicting unseen values. XGBoost is described in the paper here:

https://arxiv.org/abs/1603.02754

And the code is cited here:

Chen T, He T, Benesty M, Khotilovich V, Tang Y, Cho H, Chen K, Mitchell R, Cano I, Zhou T, Li M, Xie J, Lin M, Geng Y, Li Y, Yuan J, Cortes D (2026). xgboost: Extreme Gradient Boosting. R package version 3.5.0.0, https://github.com/dmlc/xgboost. 

In [ ]:
from sklearn.metrics import r2_score, roc_curve
from sklearn.model_selection import GroupKFold, GridSearchCV
from xgboost.sklearn import XGBRegressor

To get the best model possible, while ensuring we get effective extrapolation, we will perform a GridSearchCV to pick our XGBoost paramters. This means we will train models on 80% of the training data and predict on 20% until all data have been in the 20%. We score the out-of-fold predictions on the 20% and whatever parameters give the best score, we will use those going forward.

In [ ]:
original_features = [
    c for c in original_features
    if c in df_tf.columns and pd.api.types.is_numeric_dtype(df_tf[c])
]
X_train = df_tf[original_features]
y_train = df_tf[target]

gkf = GroupKFold(n_splits=5, shuffle=True, random_state=41)
folds = gkf.split(X_train, None, groups=df_tf['group'])

xgb = XGBRegressor(objective="reg:squarederror",
                   learning_rate=0.01,
                   random_state=3,
                   n_jobs=6)

param_grid = {
    "n_estimators": [200, 400, 600, 800, 1000],
    "max_depth": [3, 5, 7, 9],
    "subsample": [0.6, 0.7, 0.8, 0.9],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9],
}

grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=folds,
    n_jobs=1,
    verbose=2,
)

grid_search.fit(X_train, y_train)

Here we will check what the best model's parameters and cross-validated RMSE are, while also checking (but not using for anything) how well it generalizes to the completely unseen test data not used in this process.

In [ ]:
from sklearn.metrics import mean_squared_error

print("Best params:", grid_search.best_params_)
print("Best CV RMSE:", -grid_search.best_score_)

X_test = df_withheld[original_features]
y_test = df_withheld[target]

best_model = grid_search.best_estimator_
preds = best_model.predict(X_test)
test_rmse = mean_squared_error(y_test, preds)
print("Test RMSE:", test_rmse)

Because these data contain a great deal of uncertainty, we will hedge against that uncertainty by training a small ensemble of models and averaging their predictions. This small model_pool class will facilitate that averaging.

In [ ]:
class model_pool:
    def __init__(self, estimators):
        self.estimators = estimators

    def predict(self, X):
        preds = [estimator.predict(X) for estimator in self.estimators]
        means = np.mean(preds, axis=0)
        return means

    def fit(self, X, y):
        for estimator in estimators:
            estimator.fit(X, y)

For simplicity, we will use the same data splits as we used in the GridSearchCV above, training one model per CV fold to keep.

In [ ]:
xgb_pool = []

X_train = df_tf[original_features]

gkf = GroupKFold(n_splits=5, shuffle=True, random_state=41)
folds = gkf.split(X_train, None, groups=df_tf['group'])

for train_idx, test_idx in folds:
    X_train = df_tf.iloc[train_idx][original_features]
    y_train = df_tf.iloc[train_idx][target]
    
    xgb = XGBRegressor(objective="reg:squarederror",
                       learning_rate=0.01,
                       random_state=3,
                       n_jobs=6,
                       **grid_search.best_params_)
    xgb.fit(X_train, y_train)
    xgb_pool.append(xgb)

We can load these 5 trained models into a model_pool, and they will be able to predict as a team based on provided inputs. Now it is time to properly take a look at the test data, since our model training is done. 

In [ ]:
pool = model_pool(xgb_pool)
preds = pool.predict(X_test)
preds

Looking at an ROC curve can tell us how well the regressor we have trained here performs as a classifier. Since the original objective was to predict whether a peptide is/isn't an activator, this brings us back to our original goal. The higher the AUC is, the better the model is performing, with 0.5 being no better than random chance and 1.0 being perfect.

We will try a range of cutoff values to see how well the model performs at detecting activators of increasing strength. The expectation is that it should be better at detecting the strongest ones.

In [ ]:
from sklearn.metrics import roc_curve, auc

activator_threshold_values = np.linspace(0.7, 1.5, 5)
# original threshold was 0.823, the average across libraries

fig = plt.figure()
fig.set_size_inches(2, 2)

for activator_threshold in activator_threshold_values:
    test_labels = (np.array(y_test) >= activator_threshold).astype(int)
    fpr, tpr, _ = roc_curve(test_labels, preds)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr,tpr,label=f"cutoff {activator_threshold:.3f}, AUC = {roc_auc:.3f}")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()

For bookeeping, we calculate classifier performance metrics for our classifier using a threshold of 0.83, which is the average activator cutoff across datasets. We saw from the AUC curves that performance improves with threshold stringency, but let's use a single threshold value for now.

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

activator_threshold = 0.83

y_true = (np.array(y_test) >= activator_threshold).astype(int)
y_pred = (np.array(preds) >= activator_threshold).astype(int)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1], zero_division=0)
performance_df = pd.DataFrame([{
            "accuracy": accuracy_score(y_true, y_pred),
            "precision_class_0": precision[0],
            "recall_class_0": recall[0],
            "f1_class_0": f1[0],
            "precision_class_1": precision[1],
            "recall_class_1": recall[1],
            "f1_class_1": f1[1],
            "auc": roc_auc_score(y_true, preds)}])

metrics_path = "/Users/JSAlamos/Documents/Shih_lab/TF_library/Jacob_notebooks/Anya_subset_xgb_plant_classifier_metrics.csv"
performance_df.to_csv(metrics_path, index=False)
performance_df


In [ ]:
!python -m ensurepip --default-pip
!{sys.executable} -m pip install shap==0.43.0

Finally, we can use SHAP (https://arxiv.org/abs/1705.07874) to determine how much each feature contributes to the final score given to each predicted value. Features that contribute substantially will spread widely on the x-axis and will clearly separate high and low values.

In [ ]:
import shap


X = df_withheld[original_features]

explainer = shap.Explainer(pool.predict, X.sample(100), max_evals=2000)

shap_values = explainer(X)

# moved the shap plot to the next cell

In [ ]:
from matplotlib.collections import PathCollection

# Beeswarm styling from Anya_repos/plantshared_activatormodel/annotatedfeaturesrun.py
SWARM_MAX_DISPLAY = 25
my_cmap = "magma"
SWARM_HEIGHT_SCALE = 1.75
figsize = (1.8, 1.8)
font_size = 7
marker_size = 5

if hasattr(shap_values, "values"):
    shap_for_plot = np.asarray(shap_values.values)
    X_for_plot = pd.DataFrame(shap_values.data, columns=X.columns, index=X.index)
else:
    shap_for_plot = np.asarray(shap_values)
    X_for_plot = X

shap.summary_plot(
    shap_for_plot,
    X_for_plot,
    plot_type="dot",
    cmap=my_cmap,
    plot_size=(figsize[0], figsize[1] * SWARM_HEIGHT_SCALE),
    max_display=SWARM_MAX_DISPLAY,
    show=False,
)

fig = plt.gcf()
for ax in fig.get_axes():
    ax.title.set_fontsize(font_size)
    ax.xaxis.label.set_fontsize(font_size)
    ax.yaxis.label.set_fontsize(font_size)
    ax.tick_params(axis="both", labelsize=font_size)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontsize(font_size)
    for text in ax.texts:
        text.set_fontsize(font_size)
    for collection in ax.collections:
        collection.set_rasterized(False)
        if isinstance(collection, PathCollection):
            collection.set_sizes([marker_size])

plt.show()


Looking at how well predictions and observations align, we can observe that, while this model does a decent job of assigning activator/non-activator labels, its ability to predict the precise enrichment score of a given peptide is limited.

In [ ]:
r2 = r2_score(y_test, preds)

fig = plt.figure()
fig.set_size_inches(2, 2)
plt.scatter(y_test, preds,s=2)
plt.xlabel('Observations')
plt.ylabel('Predictions')
plt.title(f'r2 = {r2:0.3}')
plt.xlim(-5.1,5)